In [0]:
import pandas as pd
import numpy as np

In [0]:
df = spark.table("workspace.electiva.caso_practico_ventas").toPandas()

In [0]:
df.head(10)

In [0]:
df.tail(5)

In [0]:
# Total de registros y columnas
print(f"Total de registros: {df.shape[0]}")
print(f"Total de columnas: {df.shape[1]}")

In [0]:
#Tipos de datos asignados automáticamente
df.dtypes

In [0]:
df.info()

In [0]:
# Detección de nulos: cantidad y porcentaje por columna
nulos = df.isnull().sum()
porcentaje_nulos = (df.isnull().sum() / len(df) * 100).round(2)

In [0]:
resumen_nulos = pd.DataFrame({
    'nulos': nulos,
    'porcentaje_%': porcentaje_nulos
})
print(f"Total de valores nulos en la tabla: {nulos.sum()}")
resumen_nulos

In [0]:
# Estadística descriptiva de cantidad y precio_unitario
estadisticas = df[['cantidad', 'precio_unitario']].agg(['mean', 'median', 'std', 'min', 'max'])
estadisticas

In [0]:
df.describe()

In [0]:
# Tratamiento de duplicados
print(f"Filas completamente duplicadas: {df.duplicated().sum()}")
print(f"id_transaccion repetidos: {df.duplicated(subset=['id_transaccion']).sum()}")

In [0]:
# Eliminar duplicados completos y luego los id_transaccion repetidos
df = df.drop_duplicates()
df = df.drop_duplicates(subset=['id_transaccion'], keep='first')
df = df.reset_index(drop=True)
 
print(f"Registros después de eliminar duplicados: {df.shape[0]}")

In [0]:
# Normalización de texto en metodo_pago
# Primero vemos los valores que existen
df['metodo_pago'].value_counts()

In [0]:
#Se quitan espacios y se unifican todas las variantes de "Tarjeta Credito"
df['metodo_pago'] = df['metodo_pago'].str.strip()
 
variantes_credito = (
    df['metodo_pago']
    .str.lower()
    .str.replace('é', 'e', regex=False)
    .str.contains('tarjeta') & df['metodo_pago']
    .str.lower()
    .str.replace('é', 'e', regex=False)
    .str.contains('credito')
)
df.loc[variantes_credito, 'metodo_pago'] = 'Tarjeta de Crédito'
 
df['metodo_pago'].value_counts()

In [0]:
# Gestión de nulos: cantidad con la mediana
mediana_cantidad = df['cantidad'].median()
df['cantidad'] = df['cantidad'].fillna(mediana_cantidad)
df['cantidad'] = df['cantidad'].astype(int)
print(f"Mediana usada para cantidad: {mediana_cantidad}")

In [0]:
# Gestión de nulos: satisfaccion_cliente con la moda
moda_satisfaccion = df['satisfaccion_cliente'].mode()[0]
df['satisfaccion_cliente'] = df['satisfaccion_cliente'].fillna(moda_satisfaccion)
df['satisfaccion_cliente'] = df['satisfaccion_cliente'].astype(int)
print(f"Moda usada para satisfaccion_cliente: {moda_satisfaccion}")

In [0]:
# Tratamiento de atípicos: localizar la transacción con precio_unitario = 9999
df[df['precio_unitario'] == 9999.00]

In [0]:
df['precio_unitario'] = df['precio_unitario'].replace(9999.00, np.nan)
 
df['precio_unitario'] = df['precio_unitario'].fillna(
    df.groupby('categoria_producto')['precio_unitario'].transform('mean')
)
df['precio_unitario'] = df['precio_unitario'].round(2)

In [0]:
# Verificación: ya no deben quedar nulos
df.isnull().sum()

In [0]:
# Columna derivada: monto_total
df['monto_total'] = df['cantidad'] * df['precio_unitario']
df[['cantidad', 'precio_unitario', 'monto_total']].head()

In [0]:
# Conversión temporal y extracción de día de la semana y mes
df['fecha'] = pd.to_datetime(df['fecha'])
df['dia_semana'] = df['fecha'].dt.day_name()
df['mes'] = df['fecha'].dt.month
 
df[['fecha', 'dia_semana', 'mes']].head()

In [0]:
df['rango_monto'] = np.select(
    [df['monto_total'] < 100, df['monto_total'] <= 500],
    ['Bajo', 'Medio'],
    default='Alto'
)
df['rango_monto'].value_counts()

In [0]:
df.head(10)

In [0]:
# Group By: ingresos totales y promedio de unidades por categoría
resumen_categoria = df.groupby('categoria_producto').agg(
    ingresos_totales=('monto_total', 'sum'),
    promedio_unidades=('cantidad', 'mean')
).round(2)
 
resumen_categoria

In [0]:
# Tabla dinámica: suma de monto_total, region en filas y categoria_producto en columnas
tabla_dinamica = pd.pivot_table(
    df,
    values='monto_total',
    index='region',
    columns='categoria_producto',
    aggfunc='sum',
    fill_value=0
).round(2)
 
tabla_dinamica

In [0]:
# Análisis de satisfacción: promedio por método de pago
satisfaccion_pago = df.groupby('metodo_pago')['satisfaccion_cliente'].mean().round(2)
satisfaccion_pago

In [0]:
# Filtrado multicriterio: región Norte y satisfacción >= 4
ventas_norte = df[(df['region'] == 'Norte') & (df['satisfaccion_cliente'] >= 4)]
print(f"Ventas de la región Norte con satisfacción >= 4: {ventas_norte.shape[0]}")
ventas_norte

In [0]:
# Muestreo estratificado: 10% de los registros conservando la proporción por región
muestra = df.groupby('region', group_keys=False).sample(frac=0.10, random_state=42)
 
print(f"Registros en la muestra: {muestra.shape[0]}")
muestra

In [0]:
# Verificación de que se conserva la proporción por región
comparacion = pd.DataFrame({
    'proporcion_original_%': (df['region'].value_counts(normalize=True) * 100).round(2),
    'proporcion_muestra_%': (muestra['region'].value_counts(normalize=True) * 100).round(2)
})
comparacion